In [ ]:
import os, json, math, random, re, time
import numpy as np
from typing import Dict, List
from pathlib import Path
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from openai import OpenAI
from openai import RateLimitError
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, LoraConfig, prepare_model_for_kbit_training
import torch
from dotenv import load_dotenv

In [ ]:
CAI_DIR = "sft_data_icai"
RLAIF_DIR = "rlaif_data_cai"
PREF_PROMPTS_PATH = os.path.join(CAI_DIR, "alert_pref_prompts.jsonl")  

BASE_MODEL = "mistralai/Mistral-7B-v0.1"
FT_MODEL_DIR = "models_icai/mistral-base_v01-sft"

ANSWER_TOKENS  = 512
TEMP_A         = 0.6
TEMP_B         = 0.9
TOP_P          = 0.95

max_examples    = 1000       
seed            = 42

openai_model    = "gpt-4o-mini" 

principles_path = "icai_principles.json"

out_gen_data = os.path.join(RLAIF_DIR,f"prompts_and_generated_reponses_{max_examples}_icai_principles.json")
out_with_preference = os.path.join(RLAIF_DIR,f"gpt_preference_data_{max_examples}_icai_principles.json")
out_gen_meta = os.path.join(RLAIF_DIR,"gen_meta.json")

random.seed(seed)

# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)

In [3]:
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token  

base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=(torch.bfloat16 if torch.cuda.is_available() else torch.float32),
    device_map="auto",
)
base.config.pad_token_id = tok.pad_token_id
base.generation_config.pad_token_id = tok.pad_token_id
base.generation_config.eos_token_id = tok.eos_token_id

ft = PeftModel.from_pretrained(base, FT_MODEL_DIR)  
ft.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [4]:
def load_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

In [5]:
def load_principles(path: str) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        principles = []
        for d in data:
            principles.append(d["principle"])
    return principles

principles = load_principles(principles_path)
principles

['Select the response that addresses user concerns more thoroughly.',
 'Select the response that provides more detailed explanations.',
 'Select the response that provides detailed explanations and examples.',
 'Select the response that maintains a consistent tone throughout.',
 'Select the response that addresses user inquiry comprehensively.',
 'Select the response that provides a more detailed argument.',
 'Select the response that provides a more detailed explanation.',
 'Select the response that provides a more detailed analysis.',
 'Select the response that maintains a supportive and informative tone.',
 'Select the response that demonstrates clarity and specificity in content.',
 'Select the response that organizes information in a clearer format.',
 'Select the response that follows user instructions accurately.',
 "Select the response that follows the user's request clearly.",
 'Select the response that provides a more engaging tone.',
 'Select the response that is concise and

In [6]:
def _gen(model, prompt: str, *, temperature: float, top_p: float, max_new_tokens: int) -> str:
    if getattr(tok, "chat_template", None):
        input_ids = tok.apply_chat_template(
            [{"role": "user", "content": prompt}],
            add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)
        attention_mask = torch.ones_like(input_ids)
    else:
        encoded = tok(prompt, return_tensors="pt")
        input_ids = encoded["input_ids"].to(model.device)
        attention_mask = encoded["attention_mask"].to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )

    gen = out[0][input_ids.shape[-1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()


def generate_two_answers_with_sl_and_prompt(
    prompts,             
    out_path: str,
    write_mode: str = "w",
):
    """
    Generate A/B for all prompts using SL model (ft) and dump once as a JSON array.
    """
    norm = []
    for p in prompts:
        if isinstance(p, str):
            s = p.strip()
        else:
            s = str(p.get("prompt", "")).strip()
        if s:
            norm.append(s)

    records = []
    for user_prompt in norm:
        A = _gen(ft, user_prompt, temperature=TEMP_A, top_p=TOP_P, max_new_tokens=ANSWER_TOKENS)
        B = _gen(ft, user_prompt, temperature=TEMP_B, top_p=TOP_P, max_new_tokens=ANSWER_TOKENS)

        records.append({
            "prompt": user_prompt,
            "A": A,
            "B": B,
        })

        gen_meta = {
            "gen_meta": {
                "model_dir": FT_MODEL_DIR,
                "temp_a": TEMP_A,
                "temp_b": TEMP_B,
                "top_p": TOP_P,
            }
        }
        
    with open(out_gen_meta, "w") as f:
        json.dump(gen_meta, f, indent=4)
    
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, write_mode, encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

    print(f"[write] candidates -> {out_path} (count={len(records)})")



In [ ]:
RPM_BUDGET = 450            
SPACING = 60.0 / RPM_BUDGET 

RE_MS = re.compile(r"try again in (\d+)ms", re.I)

def call_with_retry(**kwargs):
    time.sleep(SPACING)

    backoff = 0.5       
    max_backoff = 20.0   
    max_retries = 8

    for attempt in range(max_retries):
        try:
            return client.responses.create(**kwargs)
        except RateLimitError as e:
            msg = getattr(e, "message", "") or str(e)
            m = RE_MS.search(msg)
            if m:
                wait = max(0.001, int(m.group(1)) / 1000.0)
            else:
                wait = min(max_backoff, backoff * (1.6 ** attempt)) * (0.8 + 0.4 * random.random())
            time.sleep(wait)
            continue 
        except Exception:
            raise  
    raise RuntimeError("Too many rate-limit retries; aborting.")

In [8]:
def load_generated_pairs(path: str):
    out = []
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
        for d in data:
            out.append(d)
    return out


In [9]:
def feedback_prompt(principle: str, prompt: str, respA: str, respB: str, use_principles: bool = True) -> str:
    base = (
        "Consider the following conversation between a human and an assistant.\n"
        "The goal is to evaluate which assistant response better aligns with a specific guiding principle.\n"
        "Consider the following conversation between a human and an assistant.\n"
        "The goal is to evaluate which assistant response better aligns with a specific guiding principle.\n"
        "[HUMAN/ASSISTANT CONVERSATION]\n"
        f"[HUMAN] {prompt}\n"
        f"[ASSISTANT A] {respA}\n"
        f"[ASSISTANT B] {respB}\n"
        "[PRINCIPLE FOR MULTIPLE CHOICE EVALUATION]\n"
        f"{principle}\n"
        "Options:\n"
        "  (A) [RESPONSE A]\n"
        "  (B) [RESPONSE B]\n"
        "The answer is:"
    )
    return base


In [10]:
_AB_RE = re.compile(r'[\s\(\[\{]*([AB])(?![A-Z0-9])', re.I)

def choose_label_openai(prompt_text: str):
    """
    Calls OpenAI Responses and returns (label, raw_text).
    label is 'A', 'B', or None if neither is detected.
    """
    resp = call_with_retry(
        model=openai_model,
        input=prompt_text,
        max_output_tokens=32, 
        temperature=0
    )
    try:
        txt = resp.output_text
    except Exception:
        try:
            txt = resp.output[0].content[0].text
        except Exception:
            txt = ""

    m = _AB_RE.search((txt or ""))
    label = m.group(1).upper() if m else None
    return label, (txt or "")


In [11]:
rng = np.random.default_rng(seed)

def generate_dataset(out_json: str, candidates_path: str):
    """
    Load (prompt, A, B, principle) from `candidates_path` JSONL,
    ask the GPT judge using EXACT feedback_prompt, and save labeled JSON.
    """
    # Load pre-generated pairs
    pairs = load_generated_pairs(candidates_path)
    print(f"[data] Loaded {len(pairs)} candidates from {candidates_path}")

    records = []
    for rec in tqdm(pairs, total=len(pairs), desc="Labeling"):
        prompt = rec["prompt"]
        respA  = rec["A"]
        respB  = rec["B"]

        p_idx = int(rng.integers(low=0, high=len(principles)))
        p_text = principles[p_idx]
        

        fb = feedback_prompt(
            principle=p_text,
            prompt=prompt,
            respA=respA,
            respB=respB,
        )

        label, judge_raw = choose_label_openai(fb)

        records.append({
            "prompt": prompt,
            "A": respA,
            "B": respB,
            "label": label,
            "principle": p_text,
            "feedback_model": openai_model,
            "judge_output": judge_raw
        })

    clean = [r for r in records if r["label"] in ("A", "B")]
    undecidable = [r for r in records if r["label"] not in ("A", "B")]
    print(f"kept: {len(clean)} | skipped (None): {len(undecidable)}")

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(clean, f, indent=2, ensure_ascii=False)

    audit_path = out_json.replace(".json", "_undecidable.json")
    with open(audit_path, "w", encoding="utf-8") as f:
        json.dump(undecidable, f, indent=2, ensure_ascii=False)

    clean = [r for r in records if r["label"] in ("A", "B")]
    undecidable = [r for r in records if r["label"] not in ("A", "B")]

    print(f"Saved labeled dataset to: {out_json}")
    print(f"Counts -> kept: {len(clean)} | undecidable: {len(undecidable)} | total: {len(records)}")


In [12]:
pref_items = load_jsonl(PREF_PROMPTS_PATH)
prompts = [it["prompt"] for it in pref_items if "prompt" in it and it["prompt"].strip()]
print(f"Using {len(prompts)} prompts from: {PREF_PROMPTS_PATH}")



Using 1000 prompts from: sft_data_icai/alert_pref_prompts.jsonl


In [ ]:
generate_two_answers_with_sl_and_prompt(prompts, out_path=out_gen_data)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [ ]:
generate_dataset(out_json=out_with_preference, candidates_path=out_gen_data)